# 02 - Protótipo da Base de Treino

Objetivo deste notebook: testar a preparação da base que será usada em um treino futuro, sem treinar o modelo nesta etapa.

A entrada é a tabela `training_data` salva no SQLite. Ela já contém pares `user_id` e `product_id`, sinais de afinidade, dados do usuário, dados do produto e a coluna `target`.

## 1. Configuração inicial

Carregamos as bibliotecas, definimos os caminhos principais e separamos as colunas por papel: identificadores, variáveis binárias, variáveis numéricas, categorias e alvo. Também importamos as classes do `ml_prep_kit` para reutilizar a leitura do SQLite, validações simples e o pré-processamento das features.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Define caminhos de forma flexível para rodar o notebook da raiz ou da pasta notebooks.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATABASE_PATH = PROJECT_ROOT / "data" / "training_data.db"
ML_PREP_KIT_SRC = PROJECT_ROOT / "ml_prep_kit" / "src"

# Usa o ml_prep_kit como camada reutilizável para SQLite, validação e preparo das features.
if str(ML_PREP_KIT_SRC) not in sys.path:
    sys.path.insert(0, str(ML_PREP_KIT_SRC))

from ml_prep_kit import DataValidator, FeaturePreprocessor, SQLiteDataFrameStore

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)

store = SQLiteDataFrameStore(DATABASE_PATH)
validator = DataValidator()


In [ ]:
# Parâmetros principais do protótipo.
TABLE_NAME = "training_data"
SAMPLE_USER_LIMIT = 5000
RANDOM_STATE = 42

# IDs ajudam a rastrear a recomendação, mas não entram como features.
metadata_columns = ["user_id", "product_id", "aisle_id", "department_id"]
target_column = "target"

# Flags já estão em formato 0/1 e podem entrar diretamente no modelo.
binary_columns = [
    "candidate_from_history",
    "candidate_from_cooccurrence",
    "candidate_from_favorite_category",
    "candidate_from_similar_users",
    "candidate_was_previously_purchased",
    "candidate_is_new_product_for_user",
    "is_favorite_department",
    "is_favorite_aisle",
]

# Variáveis numéricas serão preenchidas e normalizadas entre 0 e 1.
numeric_columns = [
    "cooccurrence_score",
    "category_popularity_score",
    "similar_user_score",
    "purchase_count",
    "reorder_rate",
    "avg_cart_position",
    "first_order_number",
    "last_order_number",
    "orders_since_last_purchase",
    "purchase_frequency",
    "user_total_orders",
    "user_total_items",
    "user_unique_products",
    "user_reorder_rate",
    "user_avg_cart_position",
    "user_avg_days_between_orders",
    "user_avg_order_hour",
    "user_avg_basket_size",
    "product_total_orders",
    "product_unique_users",
    "product_reorder_rate",
    "product_avg_cart_position",
    "user_department_purchase_count",
    "user_department_purchase_rate",
    "user_aisle_purchase_count",
    "user_aisle_purchase_rate",
]

# Categorias textuais serão transformadas em colunas binárias pelo preprocessador.
categorical_columns = ["aisle", "department"]

selected_columns = metadata_columns + binary_columns + numeric_columns + categorical_columns + [target_column]

DATABASE_PATH


## 2. Leitura da base

Primeiro conferimos o volume total e a distribuição do alvo usando `SQLiteDataFrameStore`. Depois carregamos uma amostra por usuário para manter o teste rápido e preservar vários produtos do mesmo cliente.


In [ ]:
# Consulta o volume total e a distribuição do alvo sem carregar a tabela inteira.
total_rows = store.summarize_table(TABLE_NAME)
target_distribution = store.summarize_table(TABLE_NAME, target_column)

display(total_rows)
display(target_distribution)


In [ ]:
# Carrega somente as colunas usadas neste protótipo.
sample_filter = f"""
user_id IN (
    SELECT DISTINCT user_id
    FROM {TABLE_NAME}
    LIMIT {SAMPLE_USER_LIMIT}
)
"""

data = store.load_dataframe(
    table_name=TABLE_NAME,
    columns=selected_columns,
    where=sample_filter,
)

data.head()


## 3. Validações rápidas

Antes de transformar os dados, verificamos tamanho da amostra, nulos, duplicados, distribuição do alvo e participação de cada fonte de recomendação.

In [ ]:
# Visão geral da amostra carregada.
summary = validator.summarize(data, "sample")
summary["users"] = data["user_id"].nunique()
summary["products"] = data["product_id"].nunique()
summary["target_rate"] = data[target_column].mean()
summary["new_product_rate"] = data["candidate_is_new_product_for_user"].mean()

summary


In [ ]:
# Valida se há nulos, duplicados ou target fora do esperado.
target_check = validator.validate_binary_column(data, target_column)

validation_report = pd.DataFrame([
    {
        "check": "duplicated_user_product",
        "rows": data.duplicated(["user_id", "product_id"]).sum(),
    },
    {"check": "invalid_target", "rows": target_check["invalid_rows"]},
    {"check": "total_nulls", "rows": data.isna().sum().sum()},
])

source_rate = data[binary_columns].mean().rename("rate").to_frame()

display(validation_report)
display(source_rate)


## 4. Separação das colunas

Os identificadores ficam separados para rastreabilidade. As features usadas no preparo são as flags binárias, as variáveis numéricas e as categorias textuais.

In [ ]:
# X contém apenas variáveis explicativas; y contém o alvo; metadata guarda rastreabilidade.
feature_columns = binary_columns + numeric_columns + categorical_columns

X = data[feature_columns]
y = data[target_column]
metadata = data[metadata_columns]

X.head()

## 5. Limpeza e normalização com ml_prep_kit

Usamos `FeaturePreprocessor`, do `ml_prep_kit`, para aplicar uma preparação reprodutível com scikit-learn. O `fit_prepare` aprende regras apenas no treino; a validação recebe as mesmas transformações com `prepare`.


In [ ]:
# Divide a amostra mantendo a proporção do target.
X_train, X_valid, y_train, y_valid, metadata_train, metadata_valid = train_test_split(
    X,
    y,
    metadata,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_valid.shape

In [ ]:
# Prepara features binárias, numéricas e categóricas com a classe reutilizável.
preprocessor = FeaturePreprocessor(
    numeric_columns=numeric_columns,
    categorical_columns=categorical_columns,
    binary_columns=binary_columns,
    min_max_range=(0, 1),
)


In [ ]:
# Aprende as regras no treino e aplica as mesmas regras na validação.
X_train_prepared = preprocessor.fit_prepare(X_train)
X_valid_prepared = preprocessor.prepare(X_valid)

X_train_prepared.head()

## 6. Validação da base preparada

Depois da transformação, verificamos se a matriz final está limpa, se as numéricas ficaram na escala esperada e se as categorias foram convertidas em colunas binárias.

In [ ]:
# Confirma se a matriz final está pronta para um treino futuro.
prepared_report = pd.DataFrame([
    {
        "dataset": "train",
        "rows": len(X_train_prepared),
        "columns": X_train_prepared.shape[1],
        "nulls": X_train_prepared.isna().sum().sum(),
        "infinite_values": np.isinf(X_train_prepared.to_numpy()).sum(),
        "min_value": X_train_prepared.min().min(),
        "max_value": X_train_prepared.max().max(),
        "target_rate": y_train.mean(),
    },
    {
        "dataset": "valid",
        "rows": len(X_valid_prepared),
        "columns": X_valid_prepared.shape[1],
        "nulls": X_valid_prepared.isna().sum().sum(),
        "infinite_values": np.isinf(X_valid_prepared.to_numpy()).sum(),
        "min_value": X_valid_prepared.min().min(),
        "max_value": X_valid_prepared.max().max(),
        "target_rate": y_valid.mean(),
    },
])

prepared_report

In [ ]:
# As colunas categóricas codificadas devem conter apenas 0 e 1.
encoded_columns = X_train_prepared.filter(regex="^cat__").columns

binary_encoding_report = pd.DataFrame([
    {
        "encoded_categorical_columns": len(encoded_columns),
        "only_binary_values": X_train_prepared[encoded_columns].isin([0, 1]).all().all(),
    }
])

binary_encoding_report

In [ ]:
# Base de protótipo pronta para inspeção ou treino futuro.
train_dataset_prototype = pd.concat(
    [
        metadata_train.reset_index(drop=True),
        X_train_prepared.reset_index(drop=True),
        y_train.reset_index(drop=True).rename("target"),
    ],
    axis=1,
)

train_dataset_prototype.head()

## 7. Conclusão

Este notebook valida uma preparação simples da base para treino futuro. A base parte do SQLite, separa identificadores e alvo, preserva flags binárias de origem da recomendação, normaliza variáveis numéricas e transforma `aisle` e `department` em colunas binárias usando `FeaturePreprocessor`, do `ml_prep_kit`.

O próximo passo será decidir quais features ficam na versão final e transformar este protótipo em um pipeline reutilizável.
